# MMAI 894 · 311 Intake Agent · Synthetic data generation (W2 Data)

Adapted from the course reference pipeline (`synth_data_pipeline.py`, airline example) to the team's Municipal 311 agent. The pipeline code lives in `synth_311_pipeline.py` (in the repo under `src/data_gen/`); step 2 loads that file, so the notebook and the repo always run the same code.

**Before you run (Kaggle right-hand panel):**
1. **Settings → Internet → On** (needs a phone-verified Kaggle account).
2. **Add-ons → Secrets** → add `OPENROUTER_API_KEY` (your `sk-or-...` key, with a credit limit set on openrouter.ai) and tick it for this notebook. The key is read at runtime and never printed.
3. Keep this notebook **private**.

Without a key, everything still runs in **offline mode** (template stand-in, no cost), so you can test the plumbing first.

| Step | What it does | Cost |
|---|---|---|
| 1-3 | Install, write the pipeline file, load the key | free |
| 4 | Offline smoke test (120 messages) | free |
| 5 | Inspect one prompt | free |
| 6 | **Pilot**: 60 messages with real models | a few cents |
| 7 | Inspect the pilot: labels, rejects, diversity | free |
| 8 | **Full run**: about 3,000 messages (off until you set `RUN_FULL = True`) | about $1-2 |
| 9 | Re-check against protected hand-written tests (later) | free (cached) |
| 10 | Zip the outputs to download | free |

## 1. Install packages

In [ ]:
!pip install -q faker langdetect

## 2. Get the pipeline file
The notebook does not carry its own copy of the code. Step 2 uses `synth_311_pipeline.py` from, in order:
1. the notebook's own folder (running locally from `src/data_gen/`);
2. a Kaggle dataset added as input (upload the file from the repo as a **private** dataset);
3. the repo on GitHub (`PIPELINE_URL`, needs Internet on, and the repo must be public or the URL must carry a token).

To change the pipeline, edit the file in the repo, not here. The cell prints the file's hash and prompt version so each run records which code it used.

In [ ]:
import glob, hashlib, os, shutil, urllib.request

PIPELINE = "synth_311_pipeline.py"
PIPELINE_URL = ("https://raw.githubusercontent.com/TerraIncognito-sudo/311_Agent_MMAI/"
                "main/src/data_gen/synth_311_pipeline.py")   # pin a commit hash instead of main for the full run

if os.path.exists(PIPELINE):
    print("Using", PIPELINE, "from the notebook folder")
else:
    found = glob.glob("/kaggle/input/*/" + PIPELINE) + glob.glob("/kaggle/input/*/*/" + PIPELINE)
    if found:
        shutil.copy(found[0], PIPELINE)
        print("Copied", found[0])
    else:
        urllib.request.urlretrieve(PIPELINE_URL, PIPELINE)
        print("Downloaded", PIPELINE_URL)

with open(PIPELINE, "rb") as f:
    print("sha256:", hashlib.sha256(f.read()).hexdigest()[:16])


## 3. Import and load the key

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
import synth_311_pipeline as sp
importlib.reload(sp)
pd.set_option("display.max_colwidth", 120)

API_KEY, SOURCE = sp.load_openrouter_key()
if API_KEY:
    print("LIVE: key loaded from", SOURCE, "(value hidden)")
else:
    print("OFFLINE: no OPENROUTER_API_KEY found - steps 6 and 8 will use the free template stand-in.")
print("prompt:", sp.PROMPT_VERSION, "| city:", sp.CITY_NAME, "| supported languages:", sorted(sp.SUPPORTED_LANGS))
print("models:", sp.MODELS)

## 4. Offline smoke test (free)
Checks the plumbing: ticket store, sampler, QC, split, review sheet, manifest. Offline messages are rigid templates, so many near-duplicate rejects are expected here.

In [ ]:
sp.main(["--offline", "--n", "120", "--out", "data/smoke"])

## 5. Look at one real prompt
What the model is asked to write for one scenario. The gold label comes from the scenario, not from the model.

In [ ]:
jobs = sp.sample_jobs(300, sp.Cfg())
job = next(j for j in jobs if j["subtype"] == "e2_trigger")
print({k: job[k] for k in ["service_code", "subtype", "expected_action", "rule_id", "hazard_flag", "lang", "model", "temperature"]})
print(sp.build_messages(job)[1]["content"])

## 6. Pilot with real models (~60 messages)
Read **all** of these with the team before the full run. The budget guard stops the run at $0.25.

In [ ]:
print("Estimated cost (upper bound): $", round(sum(sp.estimate_cost(60 // len(sp.MODELS), m) for m in sp.MODELS), 4))
sp.main(["--n", "60", "--out", "data/pilot", "--max-cost", "0.25"])

## 7. Inspect the pilot

In [ ]:
kept = pd.read_json("data/pilot/kept.jsonl", lines=True)
rej = pd.read_json("data/pilot/rejected.jsonl", lines=True)
print("kept", len(kept), "| rejected", len(rej))
display(kept[["service_code", "expected_action", "difficulty", "subtype", "lang", "model", "temperature", "message"]])

In [ ]:
# What was thrown out, and why (evidence for the report)
if len(rej):
    display(rej[["subtype", "model", "temperature", "qc_reasons", "raw_message"]])

In [ ]:
# Coverage and diversity, overall and by model / temperature / language
display(kept.groupby(["difficulty", "expected_action"]).size().rename("n").reset_index())
display(sp.diversity_report(kept))
for col in ["model", "temperature", "lang"]:
    display(sp.diversity_report(kept, [col]))

In [ ]:
# Rows where the generating model disagreed with the gold label: check these first
flagged = kept[kept["review_flags"].astype(str) != "[]"]
display(flagged[["service_code", "expected_action", "subtype", "review_flags", "message"]])

## 7b. Targeted pilot of the newer subtypes
A 60-message pilot samples only about 20 scenarios, so rare subtypes (status checks, private property, a trigger with a vague location) may not appear. This cell writes 3 different scenarios of each (about 21 calls, under $0.01) to `data/targeted_pilot.csv`. Read them before the full run.

In [ ]:
# Targeted pilot: 3 messages for each subtype that is new in 311-v4 (about 21 calls, under $0.01)
NEW = ["status_check", "status_no_postal", "status_wrong_postal", "status_new_problem",
       "trigger_vague_location", "private_tree", "private_dumping"]
cfg = sp.Cfg()
jobs = [j for j in sp.sample_jobs(3000, cfg) if j["subtype"] in NEW]
pick = []
for s in NEW:                       # 3 different scenarios per subtype, one paraphrase each
    seen = set()
    for j in jobs:
        if j["subtype"] == s and j["template_id"] not in seen and len(seen) < 3:
            seen.add(j["template_id"]); pick.append(j)
client = sp.LLMClient(api_key=API_KEY, budget=sp.Budget(max_cost_usd=0.10, max_calls=40))
res = sp.generate_dataset(pick, client, cfg)
cols = ["subtype", "service_code", "expected_action", "rule_id", "trigger_id", "ticket_ref", "model", "message"]
out = pd.concat([res["kept"].assign(kept=True), res["rejected"].assign(kept=False)])
out[cols + ["kept", "qc_reasons"]].to_csv("data/targeted_pilot.csv", index=False)
display(out[cols + ["kept", "qc_reasons"]])

## 8. Full run (~3,000 messages)
Set `RUN_FULL = True` only after the team has read the pilot. Makes about 3,000 calls one at a time (1-2 hours); keep the tab open. The cache means an interrupted run resumes without paying twice **in the same session**; click **Save Version** to keep the cache and outputs across sessions.

`--seal-adversarial` moves the adversarial rows of the test split out of `test.jsonl` into `data/protected/adversarial_sealed.jsonl` (plus a review CSV). That sealed set is for the final evaluation only: do not tune prompts or models on it, and do not commit it.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    sp.main(["--n", "3000", "--out", "data/synthetic", "--seal-adversarial", "--max-cost", "3.00", "--max-calls", "3500"])
else:
    print("Skipped. Set RUN_FULL = True to run.")

## 8b. Trigger top-up (data 5.1)
Adds about 420 trigger rows to the full run in `data/synthetic`: for each of the 20 triggers, 4 must-escalate and 3 look-alike scenarios, 3 paraphrases each (about $0.07). Existing rows keep their train/test/sealed place; the new rows are split so every trigger has both kinds in train and in test, and get their own review sheet `review_topup_5.1.csv`. Run it once; a second run is refused.

If this session no longer has `data/synthetic`, add the downloaded `outputs.zip` (unzipped) as a private Kaggle dataset and set `TOPUP_SOURCE` to its `data` folder.

In [ ]:
RUN_TOPUP = False                 # set True to add the trigger rows
TOPUP_SOURCE = None               # e.g. "/kaggle/input/311-synthetic-v5/data" if data/synthetic is gone

import os, shutil
if RUN_TOPUP:
    if not os.path.exists("data/synthetic/run_manifest.json"):
        assert TOPUP_SOURCE, "No data/synthetic in this session: set TOPUP_SOURCE"
        shutil.copytree(TOPUP_SOURCE, "data", dirs_exist_ok=True)
    sp.main(["--topup-triggers", "--out", "data/synthetic",
             "--max-cost", "0.50", "--max-calls", "600"])
else:
    print("Skipped. Set RUN_TOPUP = True to add the trigger rows.")

## 9. Later: re-check against protected hand-written tests
Upload the hand-written test inputs (John's, Uday's, Abi's, Saif's) as `.txt` files (one message per line) in a **private** Kaggle dataset, add it as input, and set the path below. Re-running the same command reuses cached model responses, so it is free and only re-applies QC (including the `matches_protected_test` leakage check) and the split.

In [ ]:
PROTECTED_DIR = None   # e.g. "/kaggle/input/311-protected-tests"
if PROTECTED_DIR:
    sp.main(["--n", "3000", "--out", "data/synthetic", "--protected-dir", PROTECTED_DIR,
             "--seal-adversarial",
             "--max-cost", "3.00", "--max-calls", "3500"])
else:
    print("Skipped. Set PROTECTED_DIR once the hand-written tests are uploaded.")

## 10. Zip the outputs
Download the zip from the **Output** panel and unzip into `Project/311/data_gen/data/`. Do not commit generated data or keys to GitHub.

In [ ]:
import shutil
shutil.make_archive("outputs", "zip", "data")
print("Wrote outputs.zip:", os.path.getsize("outputs.zip"), "bytes")